In [1]:
cargar_sql = True

In [2]:
import pandas as pd

from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

!pip install boto3
import boto3
import json
import io
# import os
# from datetime import datetime

!pip install pyathena
from pyathena import connect

from datetime import datetime, timedelta
hoy_formateado = datetime.today().strftime('%Y-%m-%d')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.0/191.0 kB 2.4 MB/s eta 0:00:00


In [3]:
txt_credenciales_athena =  r"credenciales actualizado.txt" # arrastrar json de credenciales

In [4]:
#%% Credenciales de AmazonAthena
with open('/content/' + txt_credenciales_athena) as f:
    creds = json.load(f)

conn = connect(
    aws_access_key_id     = creds["AccessKeyId"],
    aws_secret_access_key = creds["SecretAccessKey"],
    aws_session_token     = creds["SessionToken"],
    s3_staging_dir        = creds["s3_staging_dir"],
    region_name           = creds["region_name"]

    )

In [5]:
query1 = '''select * from prod_datalake_sandbox."ba__fac_line_executives"  '''
query2 = '''select * from prod_datalake_sandbox."ba__fac_line_relation"  '''
query3 = '''select * from prod_datalake_sandbox."ba__fac_line_page"  '''



cursor = conn.cursor()
cursor.execute(query1)
# Obtener los resultados
resultados = cursor.fetchall()
# Obtener los nombres de las columnas
column_names = [desc[0] for desc in cursor.description]
# Convertir los resultados a un DataFrame de pandas
df_exe = pd.DataFrame(resultados, columns = column_names)


cursor = conn.cursor()
cursor.execute(query2)
# Obtener los resultados
resultados = cursor.fetchall()
# Obtener los nombres de las columnas
column_names = [desc[0] for desc in cursor.description]
# Convertir los resultados a un DataFrame de pandas
df_rela = pd.DataFrame(resultados, columns = column_names)


cursor = conn.cursor()
cursor.execute(query3)
# Obtener los resultados
resultados = cursor.fetchall()
# Obtener los nombres de las columnas
column_names = [desc[0] for desc in cursor.description]
# Convertir los resultados a un DataFrame de pandas
df_pag = pd.DataFrame(resultados, columns = column_names)


In [6]:
# HOJA EJECUTIVO

sheet_url = 'https://docs.google.com/spreadsheets/d/1yKB117YbYMQhfL_WE3gaTFHlAlhtEizAN4Cgce6bCVc/edit?gid=106804308#gid=106804308'

spreadsheet = gc.open_by_url(sheet_url)
#Verificamos nombre de las hojas
worksheet_list = spreadsheet.worksheets()
# for i, ws in enumerate(worksheet_list):
#     print(f"Hoja {i+1}: {ws.title}")

# Seleccionamos la hoja
worksheet = spreadsheet.worksheet('Ejecutivo')

# Convertimos a df
from gspread_dataframe import get_as_dataframe
gs_ejecutivo = get_as_dataframe(worksheet, evaluate_formulas=True)
gs_ejecutivo = gs_ejecutivo.dropna(how='all')

gs_ejecutivo['_timestamp'] = pd.to_datetime(hoy_formateado)
gs_ejecutivo.columns = gs_ejecutivo.columns.str.lower()

In [7]:
# HOJA Relación

sheet_url = 'https://docs.google.com/spreadsheets/d/1yKB117YbYMQhfL_WE3gaTFHlAlhtEizAN4Cgce6bCVc/edit?gid=106804308#gid=106804308'

spreadsheet = gc.open_by_url(sheet_url)
#Verificamos nombre de las hojas
worksheet_list = spreadsheet.worksheets()
# for i, ws in enumerate(worksheet_list):
#     print(f"Hoja {i+1}: {ws.title}")

# Seleccionamos la hoja
worksheet = spreadsheet.worksheet('Relación')

# Convertimos a df
from gspread_dataframe import get_as_dataframe
gs_rel = get_as_dataframe(worksheet, evaluate_formulas=True)
gs_rel = gs_rel.dropna(how='all')

gs_rel['_timestamp'] = pd.to_datetime(hoy_formateado)
gs_rel.columns = gs_rel.columns.str.lower()

In [8]:
# HOJA Pagina

sheet_url = 'https://docs.google.com/spreadsheets/d/1yKB117YbYMQhfL_WE3gaTFHlAlhtEizAN4Cgce6bCVc/edit?gid=106804308#gid=106804308'

spreadsheet = gc.open_by_url(sheet_url)
#Verificamos nombre de las hojas
worksheet_list = spreadsheet.worksheets()
# for i, ws in enumerate(worksheet_list):
#     print(f"Hoja {i+1}: {ws.title}")

# Seleccionamos la hoja
worksheet = spreadsheet.worksheet('Pagina')

# Convertimos a df
from gspread_dataframe import get_as_dataframe
gs_page = get_as_dataframe(worksheet, evaluate_formulas=True)
gs_page = gs_page.dropna(how='all')

gs_page['_timestamp'] = pd.to_datetime(hoy_formateado)
gs_page.columns = gs_page.columns.str.lower()

In [9]:
# añadiendo datos con nuevas fechas
df_exe = df_exe[pd.to_datetime(df_exe['_timestamp'].dt.date) != pd.to_datetime(hoy_formateado)]
df_exe = pd.concat([gs_ejecutivo, df_exe], ignore_index= True)

df_rela = df_rela[pd.to_datetime(df_rela['_timestamp'].dt.date) != pd.to_datetime(hoy_formateado)]
df_rela = pd.concat([gs_rel, df_rela], ignore_index= True)

df_pag = df_pag[pd.to_datetime(df_pag['_timestamp'].dt.date) != pd.to_datetime(hoy_formateado)]
df_pag = pd.concat([gs_page, df_pag], ignore_index= True)


In [10]:
nombre_tabla = 'fac_line_executives'

# Cliente de S3
s3 = boto3.client(
    "s3",
    aws_access_key_id     = creds["AccessKeyId"],
    aws_secret_access_key = creds["SecretAccessKey"],
    aws_session_token     = creds["SessionToken"],
    region_name           = creds["region_name"]
)

# ==== CONFIGURACIÓN ====
bucket_name = "prod-datalake-sandbox-730335218320"
s3_prefix   = f"{nombre_tabla}/"  # carpeta lógica en el bucket

# ==== EXPORTAR A PARQUET EN MEMORIA ====
parquet_buffer = io.BytesIO()
df_exe.to_parquet(parquet_buffer, index=False, engine="pyarrow")
# también puedes usar engine="fastparquet" si lo prefieres

# Nombre de archivo con timestamp (opcional)
s3_key = f"{s3_prefix}{nombre_tabla}.parquet"

# Subir directamente desde el buffer
s3.put_object(
    Bucket = bucket_name,
    Key    = s3_key,
    Body   = parquet_buffer.getvalue()
)

print(f"✅ Archivo subido a s3://{bucket_name}{s3_key}")
print(f'se puede ubicar con el nombre prod_datalake_sandbox.{nombre_tabla}')

✅ Archivo subido a s3://prod-datalake-sandbox-730335218320fac_line_executives/fac_line_executives.parquet
se puede ubicar con el nombre prod_datalake_sandbox.fac_line_executives


In [11]:
nombre_tabla = 'fac_line_relation'

# Cliente de S3
s3 = boto3.client(
    "s3",
    aws_access_key_id     = creds["AccessKeyId"],
    aws_secret_access_key = creds["SecretAccessKey"],
    aws_session_token     = creds["SessionToken"],
    region_name           = creds["region_name"]
)

# ==== CONFIGURACIÓN ====
bucket_name = "prod-datalake-sandbox-730335218320"
s3_prefix   = f"{nombre_tabla}/"  # carpeta lógica en el bucket

# ==== EXPORTAR A PARQUET EN MEMORIA ====
parquet_buffer = io.BytesIO()
df_rela.to_parquet(parquet_buffer, index=False, engine="pyarrow")
# también puedes usar engine="fastparquet" si lo prefieres

# Nombre de archivo con timestamp (opcional)
s3_key = f"{s3_prefix}{nombre_tabla}.parquet"

# Subir directamente desde el buffer
s3.put_object(
    Bucket = bucket_name,
    Key    = s3_key,
    Body   = parquet_buffer.getvalue()
)

print(f"✅ Archivo subido a s3://{bucket_name}{s3_key}")
print(f'se puede ubicar con el nombre prod_datalake_sandbox.{nombre_tabla}')

✅ Archivo subido a s3://prod-datalake-sandbox-730335218320fac_line_relation/fac_line_relation.parquet
se puede ubicar con el nombre prod_datalake_sandbox.fac_line_relation


In [12]:
nombre_tabla = 'fac_line_page'

# Cliente de S3
s3 = boto3.client(
    "s3",
    aws_access_key_id     = creds["AccessKeyId"],
    aws_secret_access_key = creds["SecretAccessKey"],
    aws_session_token     = creds["SessionToken"],
    region_name           = creds["region_name"]
)

# ==== CONFIGURACIÓN ====
bucket_name = "prod-datalake-sandbox-730335218320"
s3_prefix   = f"{nombre_tabla}/"  # carpeta lógica en el bucket

# ==== EXPORTAR A PARQUET EN MEMORIA ====
parquet_buffer = io.BytesIO()
df_pag.to_parquet(parquet_buffer, index=False, engine="pyarrow")
# también puedes usar engine="fastparquet" si lo prefieres

# Nombre de archivo con timestamp (opcional)
s3_key = f"{s3_prefix}{nombre_tabla}.parquet"

# Subir directamente desde el buffer
s3.put_object(
    Bucket = bucket_name,
    Key    = s3_key,
    Body   = parquet_buffer.getvalue()
)

print(f"✅ Archivo subido a s3://{bucket_name}{s3_key}")
print(f'se puede ubicar con el nombre prod_datalake_sandbox.{nombre_tabla}')

✅ Archivo subido a s3://prod-datalake-sandbox-730335218320fac_line_page/fac_line_page.parquet
se puede ubicar con el nombre prod_datalake_sandbox.fac_line_page


In [13]:
print('fin')

fin
